In [1]:
import torch

# requires_grad=True tells PyTorch to track every operation done on this tensor,
# so it can later compute d(f)/d(a) and d(f)/d(b) automatically (autograd).
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

# f is a function of a and b. PyTorch builds a computation graph behind the
# scenes as this line runs, remembering how f was derived from a and b.
f = 3 * a**3 - b**2
f

# f is a vector (2 elements), not a single number, so .backward() needs a
# "gradient" argument telling it how to weight each output element when
# accumulating gradients (here we just weight both elements by 1).
f.backward(gradient=torch.tensor([1, 1]))

# a.grad now holds d(f)/d(a) evaluated at a = [2, 3].
# Since f = 3a^3 - b^2, d(f)/d(a) = 9a^2 -> [9*4, 9*9] = [36, 81], matching the output below.
print(a.grad)


tensor([36., 81.])


# PyTorch Day 1 — Autograd + a Binary Classifier

This notebook has two parts:

1. **Autograd demo** — shows how PyTorch automatically computes gradients (derivatives) for you, which is the mechanism neural networks use to learn.
2. **Breast cancer classifier** — a small neural network (`BCNet`) trained on the sklearn breast cancer dataset to predict malignant vs. benign.

Along the way we use `StandardScaler` from scikit-learn to normalize the input features before feeding them to the network (explained in detail below, right where it's used).


In [2]:
# --- Core PyTorch ---
import torch
import torch.nn as nn                 # neural network building blocks (layers, losses)
import torch.nn.functional as F        # functional ops (activation functions like relu/sigmoid)
import torch.optim as optim            # optimizers (e.g. Adam) that update weights during training
from torch.utils.data import DataLoader, TensorDataset  # batching/shuffling helpers for training

# --- scikit-learn: dataset + preprocessing utilities ---
from sklearn.datasets import load_breast_cancer      # a built-in toy dataset for binary classification
from sklearn.model_selection import train_test_split # splits data into train/test sets
from sklearn.preprocessing import StandardScaler     # feature scaling/normalization (see below)


In [3]:
# X = features (30 measurements per tumor, e.g. radius, texture, smoothness...)
# y = labels (0 = malignant, 1 = benign)
X, y = load_breast_cancer(return_X_y=True)

# Split into training data (used to fit the model) and test data (used to
# evaluate it on unseen examples). 20% of the data is held out for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# --- What is StandardScaler, and why do we need it? ---
# The 30 features are on very different scales (e.g. "mean area" can be in the
# hundreds, while "mean smoothness" is a small decimal like 0.1). Neural nets
# train much better/faster when every input feature has a similar scale,
# because large-scale features would otherwise dominate the gradients and
# slow down or destabilize learning.
#
# StandardScaler standardizes each feature independently to have:
#   mean = 0
#   standard deviation = 1
# using the formula: scaled_value = (x - mean) / std_dev
scaler = StandardScaler()

# .fit_transform(X_train):
#   1. "fit"       -> computes the mean and std_dev of EACH of the 30 features,
#                     but only looking at the training data.
#   2. "transform" -> applies (x - mean) / std_dev to X_train using those stats.
# fit_transform() = fit() + transform() combined into one call.
X_train_scaled = scaler.fit_transform(X_train)

# .transform(X_test):
#   Reuses the SAME mean/std_dev learned from X_train (does NOT recompute them
#   from X_test). This is important: the test set must be scaled using the
#   training set's statistics, otherwise you'd be "leaking" information from
#   the test set into preprocessing, which gives an overly optimistic
#   evaluation of the model.
#
# NOTE: the original code mistakenly passed X_train here again (a copy-paste
# bug) — that produced a 455-row array where a 114-row X_test_scaled was
# expected, causing a shape mismatch later when computing test loss/accuracy.
# Fixed below to correctly transform X_test.
X_test_scaled = scaler.transform(X_test)


In [4]:
# Peek at the scaled training data: a numpy array, shape (455, 30) — 455
# training rows x 30 standardized features. Notice values are small numbers
# centered around 0 (some negative, some positive) instead of raw measurements.
X_train_scaled


array([[-0.48368285, -0.52855104, -0.55110897, ..., -1.30018338,
        -1.5719162 , -1.3962134 ],
       [-0.50923881,  1.85103985, -0.4461158 , ...,  0.49575539,
         0.52784705,  2.17862519],
       [ 1.11214473, -0.73368818,  1.16172087, ...,  1.47788554,
         1.38437148,  1.61183973],
       ...,
       [ 0.33694735, -0.46338983,  0.46176636, ...,  1.6718677 ,
         0.59656077,  2.0968211 ],
       [-0.66257456, -0.62991292, -0.53505119, ...,  1.11588734,
         4.61711201,  1.1560741 ],
       [-1.2370157 , -0.53096442, -1.21400706, ..., -0.91802325,
        -0.5619844 ,  0.0856092 ]], shape=(455, 30))

In [5]:
# StandardScaler outputs numpy arrays, but PyTorch models need tensors, so we
# convert here. .float() ensures 32-bit floats (PyTorch's default dtype).
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()

# y_train/y_test are 1D arrays of shape (N,). .unsqueeze(1) reshapes them to
# (N, 1) — a column vector — because our model's output (and BCELoss) expects
# each label as its own row to match the network's (N, 1) output shape.
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)


In [6]:
# TensorDataset pairs up each input row with its corresponding label so they
# can be iterated together.
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

# DataLoader splits the dataset into mini-batches of 32 samples and shuffles
# the order each epoch. Training on small batches (instead of the whole
# dataset at once) is more memory-efficient and generally trains better.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


# A simple fully-connected (feedforward) neural network for binary classification.
class BCNet(nn.Module):
    def __init__(self):
        super(BCNet, self).__init__()
        # nn.Linear(in_features, out_features) applies y = xW^T + b.
        self.fc1 = nn.Linear(30, 64)   # 30 input features -> 64 hidden units
        self.fc2 = nn.Linear(64, 32)   # 64 -> 32 hidden units
        self.fc3 = nn.Linear(32, 1)    # 32 -> 1 output (probability of "benign")

    def forward(self, x):
        # ReLU adds non-linearity between layers, letting the network learn
        # more complex patterns than a plain linear model could.
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # Sigmoid squashes the final output into (0, 1), interpretable as a
        # probability — needed because we're doing binary classification.
        x = F.sigmoid(self.fc3(x))
        return x


In [7]:
model = BCNet()

# Binary Cross-Entropy loss: measures how far predicted probabilities are from
# the true 0/1 labels. Standard choice for binary classification with a
# sigmoid output.
criterion = nn.BCELoss()

# Adam optimizer: updates model.parameters() (the weights/biases) after each
# batch, using gradients computed via backprop. lr=0.001 is the step size.
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [8]:
epochs = 20  # number of full passes over the entire training dataset

for epoch in range(epochs):
    model.train()  # puts the model in "training mode" (matters for layers like dropout/batchnorm)
    running_loss = 0.0

    # DataLoader yields one shuffled mini-batch (32 samples) at a time.
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()          # clear gradients from the previous batch (they accumulate otherwise)
        preds = model(x_batch)         # forward pass: compute predictions
        loss = criterion(preds, y_batch)  # compare predictions to true labels
        loss.backward()                # backward pass: compute gradients of loss w.r.t. all weights
        optimizer.step()               # update weights using those gradients

        running_loss += loss.item()    # .item() extracts the plain Python float from the loss tensor

    # Average loss across all batches this epoch — should trend downward as
    # the model learns.
    print(f'Epoch {epoch+1}: Loss was {running_loss/len(train_loader)}')


Epoch 1: Loss was 0.6370981852213542
Epoch 2: Loss was 0.4600334405899048
Epoch 3: Loss was 0.27820809682210285
Epoch 4: Loss was 0.1764326716462771
Epoch 5: Loss was 0.12021195342143377
Epoch 6: Loss was 0.09297672733664512
Epoch 7: Loss was 0.08051542689402898
Epoch 8: Loss was 0.07228926618893941
Epoch 9: Loss was 0.06792634837329388
Epoch 10: Loss was 0.06394989751279354
Epoch 11: Loss was 0.05926827099174261
Epoch 12: Loss was 0.05451183908929427
Epoch 13: Loss was 0.05258355960249901
Epoch 14: Loss was 0.047881036251783374
Epoch 15: Loss was 0.04565926700209578
Epoch 16: Loss was 0.04281169354993229
Epoch 17: Loss was 0.040391151513904336
Epoch 18: Loss was 0.038198777784903847
Epoch 19: Loss was 0.039869787140438956
Epoch 20: Loss was 0.03590412436363598


In [9]:
# torch.no_grad() disables gradient tracking — we're only doing inference
# (evaluation), not training, so this saves memory/compute.
with torch.no_grad():
    model.eval()  # "evaluation mode": disables training-only behavior (dropout/batchnorm, if any)

    preds = model(X_test_scaled_tensor)              # predicted probabilities for the test set
    loss = criterion(preds, y_test_tensor).item()    # test loss (now that X_test_scaled is correctly sized)

    # preds >= 0.5 turns probabilities into 0/1 class predictions.
    # Compare to true labels, average the matches -> accuracy.
    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

print(f'Test loss: {loss:.4f}, Test accuracy: {accuracy:.4f}')


Test loss: 0.0596, Test accuracy: 0.9912
